In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-02-23 07:57:40--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.84.167.55, 52.84.167.134, 52.84.167.175, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.84.167.55|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  11.1MB/s    in 7.1s    

2026-02-23 07:57:48 (9.56 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



# Question 1

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()


print(f"The spark version: {spark.version}")

26/02/23 07:58:45 WARN Utils: Your hostname, DESKTOP-G33DRVE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/23 07:58:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/23 07:58:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The spark version: 3.5.0


# Question 2

In [3]:
# Read the dataset
file_path = "yellow_tripdata_2025-11.parquet"
df = spark.read.parquet(file_path)

# repartition and save as Parquet
output_path = "output/yellow_tripdata_partitioned"
df.repartition(4).write.mode("overwrite").parquet(output_path)


# Calculate the average size of Parquet files
import os

file_sizes = [os.path.getsize(os.path.join(output_path, f)) for f in os.listdir(output_path) if f.endswith(".parquet")]
avg_size_mb = sum(file_sizes) / len(file_sizes) / (1024 * 1024)
print(f"Average Parquet File Size: {avg_size_mb:.2f} MB")

[Stage 3:>                                                          (0 + 4) / 4]

Average Parquet File Size: 24.41 MB


# Question 3

In [5]:
from pyspark.sql.functions import col, to_date

# Filter trips that started on October 15th
df_filtered = df.filter(col("tpep_pickup_datetime").substr(1, 10) == "2025-11-15")

print("Trips on November 15:", df_filtered.count())

Trips on November 15: 162604


# Question 4

In [6]:
# Longest trip

df.registerTempTable('trips_data')
spark.sql("""
select MAX(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))
from trips_data
""").show()

/home/pc/spark/spark-3.5.0/python/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


+---------------------------------------------------------------------+
|max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))|
+---------------------------------------------------------------------+
|                                                                   90|
+---------------------------------------------------------------------+



In [11]:
from pyspark.sql import functions as F

df_hours = df.withColumn(
  "trip_hours",
  (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 3600.0
)

AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "CAST(tpep_dropoff_datetime AS BIGINT)" due to data type mismatch: cannot cast "TIMESTAMP_NTZ" to "BIGINT".;
'Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, cbd_congestion_fee#19, ((cast(tpep_dropoff_datetime#2 as bigint) - cast(tpep_pickup_datetime#1 as bigint)) / 3600.0) AS trip_hours#156]
+- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#

# Question 5

In [7]:
# Spark UI Port
print("Spark UI runs on port: 4040")

Spark UI runs on port: 4040


# Question 6

In [9]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-02-23 08:01:24--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.84.167.175, 52.84.167.55, 52.84.167.2, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.84.167.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-02-23 08:01:24 (11.7 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [10]:
# Least frequent pickup location zone
zone_lookup = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("trips")
zone_lookup.createOrReplaceTempView("zones")

least_frequent_zone = spark.sql("""
    SELECT zones.Zone, COUNT(*) as trip_count
    FROM trips
    JOIN zones ON trips.PULocationID = zones.LocationID
    GROUP BY zones.Zone
    ORDER BY trip_count ASC
    LIMIT 1
""").collect()[0][0]

print("Least Frequent Pickup Location Zone:", least_frequent_zone)

Least Frequent Pickup Location Zone: Governor's Island/Ellis Island/Liberty Island
